In [1]:
import pandas as pd 
alljoined = pd.read_parquet("/Users/kenantufankurt/flight-delay-pipeline/data/processed/parquet/alljoined_airlines")
alljoined.head()

,unnamed:_0,year,month,fl_date,op_unique_carrier,op_carrier_fl_num,origin_airport_id,dest_airport_id,dep_delay,dep_delay_new,...,arr_del15,arr_delay_group,cancelled,cancellation_code,diverted,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,1,2018,1,1/26/18 00:00,UA,1252,14683,12266,-7.0,0.0,...,0.0,-1.0,0,NaN,0,NaN,NaN,NaN,NaN,NaN
1,2,2018,1,1/26/18 00:00,UA,1251,10721,13930,-9.0,0.0,...,0.0,-2.0,0,NaN,0,NaN,NaN,NaN,NaN,NaN
2,3,2018,1,1/26/18 00:00,UA,1250,13930,11423,7.0,7.0,...,0.0,0.0,0,NaN,0,NaN,NaN,NaN,NaN,NaN
3,4,2018,1,1/26/18 00:00,UA,1248,11618,13930,-8.0,0.0,...,0.0,-2.0,0,NaN,0,NaN,NaN,NaN,NaN,NaN
4,5,2018,1,1/26/18 00:00,UA,1247,11278,12266,-7.0,0.0,...,0.0,0.0,0,NaN,0,NaN,NaN,NaN,NaN,NaN


In [2]:
alljoined.shape

(19174431, 24)

In [3]:
alljoined.info()

<class 'pandas.DataFrame'>
RangeIndex: 19174431 entries, 0 to 19174430
Data columns (total 24 columns):
 #   Column               Dtype  
---  ------               -----  
 0   unnamed:_0           int64  
 1   year                 int64  
 2   month                int64  
 3   fl_date              str    
 4   op_unique_carrier    str    
 5   op_carrier_fl_num    int64  
 6   origin_airport_id    int64  
 7   dest_airport_id      int64  
 8   dep_delay            float64
 9   dep_delay_new        float64
 10  dep_del15            float64
 11  dep_delay_group      float64
 12  arr_delay            float64
 13  arr_delay_new        float64
 14  arr_del15            float64
 15  arr_delay_group      float64
 16  cancelled            int64  
 17  cancellation_code    str    
 18  diverted             int64  
 19  carrier_delay        float64
 20  weather_delay        float64
 21  nas_delay            float64
 22  security_delay       float64
 23  late_aircraft_delay  float64
dtypes: floa

In [4]:
alljoined.columns

Index(['unnamed:_0', 'year', 'month', 'fl_date', 'op_unique_carrier',
       'op_carrier_fl_num', 'origin_airport_id', 'dest_airport_id',
       'dep_delay', 'dep_delay_new', 'dep_del15', 'dep_delay_group',
       'arr_delay', 'arr_delay_new', 'arr_del15', 'arr_delay_group',
       'cancelled', 'cancellation_code', 'diverted', 'carrier_delay',
       'weather_delay', 'nas_delay', 'security_delay', 'late_aircraft_delay'],
      dtype='str')

In [5]:
alljoined.duplicated(
    subset=[
        'fl_date',
        'op_unique_carrier',
        'op_carrier_fl_num',
        'origin_airport_id',
        'dest_airport_id'
    ]
).sum()

np.int64(4)

In [6]:
alljoined.groupby([
    'fl_date',
    'op_unique_carrier',
    'op_carrier_fl_num',
    'origin_airport_id',
    'dest_airport_id'
]).size().value_counts().head()

1    19174423
2           4
Name: count, dtype: int64

In [7]:
alljoined = alljoined.drop_duplicates(
    subset=[
        'fl_date',
        'op_unique_carrier',
        'op_carrier_fl_num',
        'origin_airport_id',
        'dest_airport_id'
    ]
)

In [8]:
alljoined.groupby([
    'fl_date',
    'op_unique_carrier',
    'op_carrier_fl_num',
    'origin_airport_id',
    'dest_airport_id'
]).size().value_counts().head()

1    19174427
Name: count, dtype: int64

In [9]:
df_clean = alljoined.drop(columns=[
    'unnamed:_0',
    'year',
    'month',
    'dep_delay_new',
    'dep_del15',
    'dep_delay_group',
    'arr_delay_new',
    'arr_del15',
    'arr_delay_group'
])

In [10]:
df_clean.columns

Index(['fl_date', 'op_unique_carrier', 'op_carrier_fl_num',
       'origin_airport_id', 'dest_airport_id', 'dep_delay', 'arr_delay',
       'cancelled', 'cancellation_code', 'diverted', 'carrier_delay',
       'weather_delay', 'nas_delay', 'security_delay', 'late_aircraft_delay'],
      dtype='str')

In [11]:
df_clean = df_clean.rename(columns={
    'fl_date': 'flight_date',
    'op_unique_carrier': 'airline_code',
    'op_carrier_fl_num': 'flight_number',
    'origin_airport_id': 'origin_airport_id',
    'dest_airport_id': 'destination_airport_id',
    'dep_delay': 'departure_delay',
    'arr_delay': 'arrival_delay',
    'cancelled': 'is_cancelled',
    'diverted': 'is_diverted'
})

In [12]:
df_clean.columns

Index(['flight_date', 'airline_code', 'flight_number', 'origin_airport_id',
       'destination_airport_id', 'departure_delay', 'arrival_delay',
       'is_cancelled', 'cancellation_code', 'is_diverted', 'carrier_delay',
       'weather_delay', 'nas_delay', 'security_delay', 'late_aircraft_delay'],
      dtype='str')

In [13]:
#primary key test
df_clean.duplicated(subset=[
    'flight_date',
    'airline_code',
    'flight_number',
    'origin_airport_id',
    'destination_airport_id'
]).sum()

np.int64(0)

In [14]:
df_clean[['flight_date','airline_code','flight_number',
          'origin_airport_id','destination_airport_id']].isnull().sum()

flight_date               0
airline_code              0
flight_number             0
origin_airport_id         0
destination_airport_id    0
dtype: int64

In [15]:
df_clean[['departure_delay','arrival_delay']].isnull().sum()

departure_delay    458134
arrival_delay      506605
dtype: int64

In [16]:
df_clean[(df_clean['is_cancelled'] == 0) & (df_clean['arrival_delay'].isnull())].shape

(41767, 15)

In [17]:
df_clean = df_clean[
    ~(
        (df_clean['is_cancelled'] == 0) &
        (df_clean['arrival_delay'].isnull())
    )
]

In [18]:
df_clean[
    (df_clean['is_cancelled'] == 0) &
    (df_clean['arrival_delay'].isnull())
].shape

(0, 15)

In [19]:
df_clean.dtypes

flight_date                   str
airline_code                  str
flight_number               int64
origin_airport_id           int64
destination_airport_id      int64
departure_delay           float64
arrival_delay             float64
is_cancelled                int64
cancellation_code             str
is_diverted                 int64
carrier_delay             float64
weather_delay             float64
nas_delay                 float64
security_delay            float64
late_aircraft_delay       float64
dtype: object

In [20]:
df_clean['flight_date'] = pd.to_datetime(df_clean['flight_date'])

/var/folders/cn/h7lf8gjx0j94d61z7l6_2z500000gn/T/ipykernel_44989/3580804858.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean['flight_date'] = pd.to_datetime(df_clean['flight_date'])


In [21]:
df_clean['flight_date'].isnull().sum()

np.int64(0)

In [22]:
df_clean['is_cancelled'] = df_clean['is_cancelled'].astype(bool)
df_clean['is_diverted'] = df_clean['is_diverted'].astype(bool)

In [23]:
df_clean.dtypes

flight_date               datetime64[us]
airline_code                         str
flight_number                      int64
origin_airport_id                  int64
destination_airport_id             int64
departure_delay                  float64
arrival_delay                    float64
is_cancelled                        bool
cancellation_code                    str
is_diverted                         bool
carrier_delay                    float64
weather_delay                    float64
nas_delay                        float64
security_delay                   float64
late_aircraft_delay              float64
dtype: object

In [24]:
df_clean[['departure_delay','arrival_delay']].describe()

,departure_delay,arrival_delay
count,1.867453e+07,1.866782e+07
mean,9.451768e+00,3.204798e+00
std,4.314013e+01,4.535193e+01
min,-1.280000e+02,-1.390000e+02
25%,-5.000000e+00,-1.500000e+01
50%,-2.000000e+00,-7.000000e+00
75%,7.000000e+00,6.000000e+00
max,3.890000e+03,3.864000e+03


In [26]:
from pathlib import Path

Path("data/processed/curated").mkdir(parents=True, exist_ok=True)

In [27]:
df_clean.to_parquet(
    "data/processed/curated/flight_delays_curated.parquet",
    index=False
)

In [28]:
import os
print(os.getcwd())

/Users/kenantufankurt/flight-delay-pipeline/notebooks


In [29]:
df_clean.to_parquet(
    "../data/processed/curated/flight_delays_curated.parquet",
    index=False
)

In [30]:
airline = pd.read_parquet("/Users/kenantufankurt/flight-delay-pipeline/data/processed/parquet/airline_key.parquet")
airline.head()

,unnamed:_0,op_unique_carrier,airline_name
0,1,AA,American
1,2,AS,Alaska
2,3,B6,JetBlue
3,4,DL,Delta
4,5,F9,Frontier


In [31]:
airline = airline.drop(columns=['unnamed:_0'])
airline.head()

,op_unique_carrier,airline_name
0,AA,American
1,AS,Alaska
2,B6,JetBlue
3,DL,Delta
4,F9,Frontier


In [32]:
airline = airline.rename(columns = {"op_unique_carrier":"airline_code"})

In [33]:
airline.duplicated().sum()

np.int64(0)

In [34]:
airline.duplicated(subset=['airline_code']).sum()

np.int64(0)

In [35]:
airline.isnull().sum()

airline_code    0
airline_name    0
dtype: int64

In [40]:
airline.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   airline_code  10 non-null     str  
 1   airline_name  10 non-null     str  
dtypes: str(2)
memory usage: 384.0 bytes


In [41]:
airline.head(10)

,airline_code,airline_name
0,AA,American
1,AS,Alaska
2,B6,JetBlue
3,DL,Delta
4,F9,Frontier
5,G4,Allegiant
6,HA,Hawaiian
7,NK,Spirit
8,UA,United
9,WN,Southwest


In [42]:
airline.to_parquet(
    "../data/processed/curated/airline_curated.parquet",
    index=False
)

In [43]:
airport = pd.read_parquet('/Users/kenantufankurt/flight-delay-pipeline/data/processed/parquet/airport_info.parquet')
airport.head()

,orgin_aiport_id,description,code.y
0,10001.0,"Afognak Lake, AK: Afognak Lake Airport",01A
1,10003.0,"Granite Mountain, AK: Bear Creek Mining Strip",03A
2,10004.0,"Lik, AK: Lik Mining Camp",04A
3,10005.0,"Little Squaw, AK: Little Squaw Airport",05A
4,10006.0,"Kizhuyak, AK: Kizhuyak Bay",06A


In [61]:
airport = airport.rename(columns={
    'orgin_aiport_id': 'airport_id',
    'description': 'airport_name',
    'code.y': 'airport_code'
})

In [55]:
airport.dtypes

airport_id      float64
airport_name        str
code.y              str
dtype: object

In [62]:
airport.head()

,airport_id,airport_name,airport_code
0,10001.0,"Afognak Lake, AK: Afognak Lake Airport",01A
1,10003.0,"Granite Mountain, AK: Bear Creek Mining Strip",03A
2,10004.0,"Lik, AK: Lik Mining Camp",04A
3,10005.0,"Little Squaw, AK: Little Squaw Airport",05A
4,10006.0,"Kizhuyak, AK: Kizhuyak Bay",06A


In [63]:
airport.isnull().sum()


airport_id       0
airport_name     0
airport_code    34
dtype: int64

In [59]:
airport = airport.dropna(subset=['airport_id'])

In [64]:
airport.isnull().sum()

airport_id       0
airport_name     0
airport_code    34
dtype: int64

In [65]:
airport.dtypes

airport_id      float64
airport_name        str
airport_code        str
dtype: object

In [66]:
airport['airport_id'] = airport['airport_id'].astype(int)

In [67]:
airport.dtypes

airport_id      int64
airport_name      str
airport_code      str
dtype: object

In [69]:
airport.head()

,airport_id,airport_name,airport_code
0,10001,"Afognak Lake, AK: Afognak Lake Airport",01A
1,10003,"Granite Mountain, AK: Bear Creek Mining Strip",03A
2,10004,"Lik, AK: Lik Mining Camp",04A
3,10005,"Little Squaw, AK: Little Squaw Airport",05A
4,10006,"Kizhuyak, AK: Kizhuyak Bay",06A


In [70]:
airport.duplicated(subset=['airport_id']).sum()

np.int64(72)

In [71]:
airport[airport.duplicated(subset=['airport_id'], keep=False)]\
    .sort_values('airport_id')

,airport_id,airport_name,airport_code
9,10011,"Peach Springs, AZ: Grand Canyon West",1G4
10,10011,"Peach Springs, AZ: Grand Canyon West",DQR
22,10023,"Tikchik Lodge, AK: Tikchik Lodge Airport",KTH
23,10023,"Tikchik Lodge, AK: Tikchik Lodge Airport",TLK
32,10033,"Alpine, AK: Alpine Airstrip",A20
...,...,...,...
6423,16692,"Three Forks, MT: Three Forks Airport",MT2
6434,16703,"Rivas, Nicaragua: Costa Esmeralda",ECI
6435,16703,"Rivas, Nicaragua: Costa Esmeralda",NI1
6563,16832,"New York, NY: New York Skyports Seaplane Base",NYS


In [72]:
airport = airport.drop_duplicates(subset = ['airport_id'], keep = 'first')

In [73]:
airport.duplicated(subset=['airport_id']).sum()

np.int64(0)

In [74]:
airport.to_parquet(
    "../data/processed/curated/airport_curated.parquet",
    index=False
)

In [15]:
import pandas as pd 
wp = pd.read_parquet("/Users/kenantufankurt/flight-delay-pipeline/data/processed/parquet/weather.parquet")
wp.head()

,time,temperature_2m_max,temperature_2m_min,precipitation_sum,snowfall_sum,rain_sum,date
0,2018-01-01,-6.5,-14.1,0.0,0.00,0.0,2018-01-01
1,2018-01-02,-2.9,-11.4,0.0,0.00,0.0,2018-01-02
2,2018-01-03,-2.2,-12.2,0.0,0.00,0.0,2018-01-03
3,2018-01-04,-4.2,-7.1,20.1,14.07,0.0,2018-01-04
4,2018-01-05,-8.0,-13.9,0.0,0.00,0.0,2018-01-05


In [16]:
wp = wp.drop(columns=['time'])

In [17]:
wp.head()

,temperature_2m_max,temperature_2m_min,precipitation_sum,snowfall_sum,rain_sum,date
0,-6.5,-14.1,0.0,0.00,0.0,2018-01-01
1,-2.9,-11.4,0.0,0.00,0.0,2018-01-02
2,-2.2,-12.2,0.0,0.00,0.0,2018-01-03
3,-4.2,-7.1,20.1,14.07,0.0,2018-01-04
4,-8.0,-13.9,0.0,0.00,0.0,2018-01-05


In [18]:
wp.columns

Index(['temperature_2m_max', 'temperature_2m_min', 'precipitation_sum',
       'snowfall_sum', 'rain_sum', 'date'],
      dtype='str')

In [19]:
wp.shape

(1673, 6)

In [20]:
wp.info()

<class 'pandas.DataFrame'>
RangeIndex: 1673 entries, 0 to 1672
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   temperature_2m_max  1673 non-null   float64       
 1   temperature_2m_min  1673 non-null   float64       
 2   precipitation_sum   1673 non-null   float64       
 3   snowfall_sum        1673 non-null   float64       
 4   rain_sum            1673 non-null   float64       
 5   date                1673 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(5)
memory usage: 78.6 KB


In [21]:
wp['date'].isnull().sum()

np.int64(0)

In [22]:
wp['date'].duplicated().sum()

np.int64(0)

In [23]:
wp['date'].min() , wp['date'].max()

(Timestamp('2018-01-01 00:00:00'), Timestamp('2022-07-31 00:00:00'))

In [29]:
flight_delays = pd.read_parquet("/Users/kenantufankurt/flight-delay-pipeline/data/processed/curated/flight_delays_curated.parquet")
flight_delays.head()

,flight_date,airline_code,flight_number,origin_airport_id,destination_airport_id,departure_delay,arrival_delay,is_cancelled,cancellation_code,is_diverted,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,2018-01-26,UA,1252,14683,12266,-7.0,-12.0,False,NaN,False,NaN,NaN,NaN,NaN,NaN
1,2018-01-26,UA,1251,10721,13930,-9.0,-26.0,False,NaN,False,NaN,NaN,NaN,NaN,NaN
2,2018-01-26,UA,1250,13930,11423,7.0,7.0,False,NaN,False,NaN,NaN,NaN,NaN,NaN
3,2018-01-26,UA,1248,11618,13930,-8.0,-27.0,False,NaN,False,NaN,NaN,NaN,NaN,NaN
4,2018-01-26,UA,1247,11278,12266,-7.0,1.0,False,NaN,False,NaN,NaN,NaN,NaN,NaN


In [28]:
wp.head()

,temperature_2m_max,temperature_2m_min,precipitation_sum,snowfall_sum,rain_sum,date
0,-6.5,-14.1,0.0,0.00,0.0,2018-01-01
1,-2.9,-11.4,0.0,0.00,0.0,2018-01-02
2,-2.2,-12.2,0.0,0.00,0.0,2018-01-03
3,-4.2,-7.1,20.1,14.07,0.0,2018-01-04
4,-8.0,-13.9,0.0,0.00,0.0,2018-01-05


In [30]:
df_final = flight_delays.merge(
wp,
left_on= 'flight_date',
right_on='date',
how='left')


In [31]:
df_final.head()

,flight_date,airline_code,flight_number,origin_airport_id,destination_airport_id,departure_delay,arrival_delay,is_cancelled,cancellation_code,is_diverted,...,weather_delay,nas_delay,security_delay,late_aircraft_delay,temperature_2m_max,temperature_2m_min,precipitation_sum,snowfall_sum,rain_sum,date
0,2018-01-26,UA,1252,14683,12266,-7.0,-12.0,False,NaN,False,...,NaN,NaN,NaN,NaN,1.6,-6.7,0.0,0.0,0.0,2018-01-26
1,2018-01-26,UA,1251,10721,13930,-9.0,-26.0,False,NaN,False,...,NaN,NaN,NaN,NaN,1.6,-6.7,0.0,0.0,0.0,2018-01-26
2,2018-01-26,UA,1250,13930,11423,7.0,7.0,False,NaN,False,...,NaN,NaN,NaN,NaN,1.6,-6.7,0.0,0.0,0.0,2018-01-26
3,2018-01-26,UA,1248,11618,13930,-8.0,-27.0,False,NaN,False,...,NaN,NaN,NaN,NaN,1.6,-6.7,0.0,0.0,0.0,2018-01-26
4,2018-01-26,UA,1247,11278,12266,-7.0,1.0,False,NaN,False,...,NaN,NaN,NaN,NaN,1.6,-6.7,0.0,0.0,0.0,2018-01-26


In [33]:
df_final.columns

Index(['flight_date', 'airline_code', 'flight_number', 'origin_airport_id',
       'destination_airport_id', 'departure_delay', 'arrival_delay',
       'is_cancelled', 'cancellation_code', 'is_diverted', 'carrier_delay',
       'weather_delay', 'nas_delay', 'security_delay', 'late_aircraft_delay',
       'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum',
       'snowfall_sum', 'rain_sum', 'date'],
      dtype='str')

In [34]:
df_final.isnull().sum()

flight_date                      0
airline_code                     0
flight_number                    0
origin_airport_id                0
destination_airport_id           0
departure_delay             458130
arrival_delay               464838
is_cancelled                     0
cancellation_code         18667822
is_diverted                      0
carrier_delay             15802257
weather_delay             15802257
nas_delay                 15802257
security_delay            15802257
late_aircraft_delay       15802257
temperature_2m_max               0
temperature_2m_min               0
precipitation_sum                0
snowfall_sum                     0
rain_sum                         0
date                             0
dtype: int64

In [37]:
flight_delays.isnull().sum()

flight_date                      0
airline_code                     0
flight_number                    0
origin_airport_id                0
destination_airport_id           0
departure_delay             458130
arrival_delay               464838
is_cancelled                     0
cancellation_code         18667822
is_diverted                      0
carrier_delay             15802257
weather_delay             15802257
nas_delay                 15802257
security_delay            15802257
late_aircraft_delay       15802257
dtype: int64

In [36]:
df_final = df_final.drop(columns=['date'])

In [39]:
df_final.columns

Index(['flight_date', 'airline_code', 'flight_number', 'origin_airport_id',
       'destination_airport_id', 'departure_delay', 'arrival_delay',
       'is_cancelled', 'cancellation_code', 'is_diverted', 'carrier_delay',
       'weather_delay', 'nas_delay', 'security_delay', 'late_aircraft_delay',
       'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum',
       'snowfall_sum', 'rain_sum'],
      dtype='str')

In [45]:
df_final.duplicated(subset=['flight_date','flight_number','origin_airport_id','destination_airport_id','airline_code']).sum()

np.int64(0)

In [46]:
df_final = df_final.rename(columns={
    "temperature_2m_max": "max_temperature_c",
    "temperature_2m_min": "min_temperature_c",
    "precipitation_sum": "total_precipitation_mm",
    "snowfall_sum": "total_snowfall_mm",
    "rain_sum": "total_rain_mm",
    "date": "weather_date"
})

In [47]:
df_final.columns

Index(['flight_date', 'airline_code', 'flight_number', 'origin_airport_id',
       'destination_airport_id', 'departure_delay', 'arrival_delay',
       'is_cancelled', 'cancellation_code', 'is_diverted', 'carrier_delay',
       'weather_delay', 'nas_delay', 'security_delay', 'late_aircraft_delay',
       'max_temperature_c', 'min_temperature_c', 'total_precipitation_mm',
       'total_snowfall_mm', 'total_rain_mm'],
      dtype='str')

In [48]:
cols = [
    "flight_date",
    "airline_code",
    "flight_number",
    "origin_airport_id",
    "destination_airport_id",
    
    "departure_delay",
    "arrival_delay",
    "carrier_delay",
    "weather_delay",
    "nas_delay",
    "security_delay",
    "late_aircraft_delay",
    
    "max_temperature_c",
    "min_temperature_c",
    "total_precipitation_mm",
    "total_snowfall_mm",
    "total_rain_mm"
]

df_final = df_final[cols]

In [ ]:
df_final.to_parquet("/Users/kenantufankurt/flight-delay-pipeline/data/processed/intermediate/flight_weather.parquet")

In [24]:
import os
print(os.getcwd())

/Users/kenantufankurt/flight-delay-pipeline/notebooks


In [25]:
import pandas as pd

holidays = holidays = pd.read_csv("../data/raw/us_holidays.csv")

print(holidays.head())

         date                holiday_name
0  2018-01-01              New Year's Day
1  2018-01-15  Martin Luther King Jr. Day
2  2018-02-19              Presidents Day
3  2018-05-28                Memorial Day
4  2018-07-04            Independence Day


In [26]:
holidays.shape

(52, 2)

In [27]:
holidays.isnull().sum()

date            0
holiday_name    0
dtype: int64

In [11]:
holidays.duplicated().sum()

np.int64(0)

In [ ]:
holidays.to_parquet("../data/processed/curated/holidays.parquet",index=False)

In [1]:
import pandas as pd 
df = pd.read_parquet("/Users/kenantufankurt/flight-delay-pipeline/data/processed/intermediate/flight_weather.parquet")
df.head()

,flight_date,airline_code,flight_number,origin_airport_id,destination_airport_id,departure_delay,arrival_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay,max_temperature_c,min_temperature_c,total_precipitation_mm,total_snowfall_mm,total_rain_mm
0,2018-01-26,UA,1252,14683,12266,-7.0,-12.0,NaN,NaN,NaN,NaN,NaN,1.6,-6.7,0.0,0.0,0.0
1,2018-01-26,UA,1251,10721,13930,-9.0,-26.0,NaN,NaN,NaN,NaN,NaN,1.6,-6.7,0.0,0.0,0.0
2,2018-01-26,UA,1250,13930,11423,7.0,7.0,NaN,NaN,NaN,NaN,NaN,1.6,-6.7,0.0,0.0,0.0
3,2018-01-26,UA,1248,11618,13930,-8.0,-27.0,NaN,NaN,NaN,NaN,NaN,1.6,-6.7,0.0,0.0,0.0
4,2018-01-26,UA,1247,11278,12266,-7.0,1.0,NaN,NaN,NaN,NaN,NaN,1.6,-6.7,0.0,0.0,0.0
